# Project 2.1: Image Classification Using Convolutional Neural Networks (CNNs)

This notebook demonstrates how to build, train, evaluate, and visualize a complete image classification system using Convolutional Neural Networks (CNNs) in **PyTorch**. We will use the standard **CIFAR-10** dataset containing 60,000 32x32 color images in 10 classes.

## 1. Import Dependencies and Setup Environment

We begin by importing the necessary libraries including PyTorch, Torchvision, Matplotlib, and Scikit-Learn.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Dataset Loading and Preprocessing

CIFAR-10 contains images with dimensions 32x32x3. To train the network effectively and prevent overfitting, we will apply:
- **Data Augmentation** on the training set (Random Crop with padding, Random Horizontal Flip, Random Rotation).
- **Normalization** on both training and test sets using the dataset-wide mean and standard deviation: Mean = `[0.4914, 0.4822, 0.4465]`, Std = `[0.2470, 0.2435, 0.2616]`.

In [ ]:
# Normalization statistics
CIFAR10_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR10_STD = [0.2470, 0.2435, 0.2616]

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

# Validation/Test transforms (only normalization)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

# Download and load CIFAR-10
data_dir = './data'
full_train_dataset = torchvision.datasets.CIFAR10(root=data_dir, train=True, download=True)
test_dataset = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=val_transform)

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print(f"Loaded {len(full_train_dataset)} train images and {len(test_dataset)} test images across {len(classes)} classes.")

### Visualize Random Samples from the Dataset

Let's look at 10 random images from the raw training dataset to get an idea of the samples.

In [ ]:
plt.figure(figsize=(12, 5))
indices = np.random.choice(len(full_train_dataset), 10, replace=False)

for i, idx in enumerate(indices):
    img, label = full_train_dataset[idx]
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(classes[label])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Explaining CNN Layers and Architecture

Before coding the model, let's review how CNN layers operate:
1.  **Convolutional Layer (`nn.Conv2d`)**: Slides a filter grid (kernel) over the image to extract local features (edges, corners, textures). Stacking layers lets the network build hierarchical features (e.g. edge -> eye -> face).
2.  **Activation Function (`nn.ReLU`)**: Introduces non-linearity ($f(x) = \max(0, x)$). Without it, the network would just be a large linear model.
3.  **Batch Normalization (`nn.BatchNorm2d`)**: Normalizes layer inputs across a batch. This stabilizes training, reduces sensitivity to initializations, and acts as a mild regularizer.
4.  **Max Pooling (`nn.MaxPool2d`)**: Downsamples spatial dimensions (e.g., $32\times32 \rightarrow 16\times16$) by picking the maximum value in local grids. This reduces calculation load and grants translation invariance.
5.  **Dropout (`nn.Dropout`)**: Randomly sets a percentage of inputs to 0 during training, preventing nodes from co-adapting and reducing overfitting.
6.  **Dense/Linear Layer (`nn.Linear`)**: Fully connected layers that map final 1D feature vectors to class score logits.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout(0.25)
        
        # Block 2
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout2 = nn.Dropout(0.25)
        
        # Block 3
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.dropout3 = nn.Dropout(0.4)
        
        # Fully Connected Block (Output dimension is 4x4 after 3 poolings of 32x32)
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.bn7 = nn.BatchNorm1d(512)
        self.dropout4 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 10)
        
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.dropout1(self.pool1(x))
        
        x = torch.relu(self.bn3(self.conv3(x)))
        x = torch.relu(self.bn4(self.conv4(x)))
        x = self.dropout2(self.pool2(x))
        
        x = torch.relu(self.bn5(self.conv5(x)))
        x = torch.relu(self.bn6(self.conv6(x)))
        x = self.dropout3(self.pool3(x))
        
        x = x.view(x.size(0), -1)
        x = torch.relu(self.bn7(self.fc1(x)))
        x = self.dropout4(x)
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(model)

## 4. Model Training Setup

We split the training data into 80% Train and 20% Validation. We'll set up the cross-entropy loss function and Adam optimizer.

In [ ]:
# Setup train/val subsets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_subset, val_subset = random_split(
    full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

# Custom wrapper dataset to apply different transforms to train/val
class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        img, label = self.subset.dataset.data[self.subset.indices[index]], self.subset.dataset.targets[self.subset.indices[index]]
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, label
    def __len__(self):
        return len(self.subset)

train_dataset_loaded = DatasetWrapper(train_subset, train_transform)
val_dataset_loaded = DatasetWrapper(val_subset, val_transform)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset_loaded, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset_loaded, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 15
print(f"Batches in train_loader: {len(train_loader)}")
print(f"Batches in val_loader: {len(val_loader)}")

### Training Loop

Let's execute the training loops. During training, the best model weights will be saved based on validation accuracy.

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
model_save_path = 'best_cnn_model.pth'

print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / total
    epoch_train_acc = 100.0 * correct / total
    
    # Validation evaluation
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = 100.0 * correct_val / total_val
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), model_save_path)
        print(f"  --> Saved new best validation accuracy: {best_val_acc:.2f}%")

## 5. Model Evaluation and Visualization

Let's visualize the loss and accuracy metrics over training epochs.

In [ ]:
# Plot Loss Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), history['train_loss'], label='Train Loss', color='indigo', marker='o')
plt.plot(range(1, epochs + 1), history['val_loss'], label='Val Loss', color='crimson', marker='o')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy Curves
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), history['train_acc'], label='Train Acc', color='teal', marker='o')
plt.plot(range(1, epochs + 1), history['val_acc'], label='Val Acc', color='forestgreen', marker='o')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Test Set Evaluation and Confusion Matrix

We'll load the best saved weights and compute the classification report and confusion matrix on the test dataset.

In [ ]:
# Load best weights
best_model = SimpleCNN().to(device)
if os.path.exists(model_save_path):
    best_model.load_state_dict(torch.load(model_save_path, map_location=device))
best_model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

print(classification_report(all_targets, all_preds, target_names=classes))

# Plot Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix on Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### Visualize Correct and Incorrect Predictions

Let's display examples of correctly and incorrectly classified images to understand where the model makes errors.

In [ ]:
# Get some test samples
images, labels = next(iter(test_loader))
images_dev = images.to(device)
with torch.no_grad():
    outputs = best_model(images_dev)
    _, preds = torch.max(outputs, 1)

preds = preds.cpu().numpy()
labels = labels.numpy()

# Helper to denormalize images for plotting
def denormalize(tensor):
    mean = np.array(CIFAR10_MEAN)
    std = np.array(CIFAR10_STD)
    img = tensor.numpy().transpose((1, 2, 0))
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

# Find indices
correct_indices = np.where(preds == labels)[0]
incorrect_indices = np.where(preds != labels)[0]

# Plot Correct ones
print("Correctly Classified Images:")
plt.figure(figsize=(12, 4))
for i in range(min(5, len(correct_indices))):
    idx = correct_indices[i]
    plt.subplot(1, 5, i+1)
    plt.imshow(denormalize(images[idx]))
    plt.title(f"True/Pred: {classes[labels[idx]]}", color='green')
    plt.axis('off')
plt.tight_layout()
plt.show()

# Plot Incorrect ones
print("Incorrectly Classified Images:")
plt.figure(figsize=(12, 4))
for i in range(min(5, len(incorrect_indices))):
    idx = incorrect_indices[i]
    plt.subplot(1, 5, i+1)
    plt.imshow(denormalize(images[idx]))
    plt.title(f"T: {classes[labels[idx]]}\nP: {classes[preds[idx]]}", color='red')
    plt.axis('off')
plt.tight_layout()
plt.show()